# Asahlagi — Fine-tune Quiz Generator (IndoT5, ANSWER-AWARE)

**Owner**: Audry (Quiz Generator)

Fine-tunes `Wikidepia/IndoT5-base` for **answer-aware** Indonesian question generation on
TyDiQA-id: the input sentence has the answer span wrapped in `<hl> … <hl>`, and the model
learns to ask a question **whose answer is that span**. Trains to 8 epochs saving a
checkpoint each epoch, so section 7 can compare epochs 4-8 from a single run.

**Why answer-aware (v2):** the previous fine-tune trained on the bare sentence (no
highlight), so it generated a question but the backend had to *guess* the answer
(`max(keywords, key=len)`) → nonsensical options. With `<hl>` the answer is chosen first and
the model is told to ask about it, so the correct answer genuinely answers the question.

**Important**
- Runtime: **GPU (T4)** (Runtime → Change runtime type → T4 GPU).
- `fp16` MUST stay **False** — T5 + fp16 caused NaN/garbage. `bf16` also False (T4 = Turing,
  no bf16 support); training runs in fp32.
- Prefix `"buat pertanyaan: "` **and** the `<hl> {answer} <hl>` format must match the HF
  Space inference side (`backend/asahlagi-quizgen/qg_core.py`).
- `<hl>` is registered as a special token (section 3) and pushed with the model, so the
  Space tokenizer recognises it.
- Model learns *highlighted passage → question*; options/distractors come from the backend.
- One full run to 8 epochs is ~1.5-2 h on a T4.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==4.45.2" "datasets==3.0.1" sentencepiece accelerate evaluate sacrebleu

## 2. Prepare data (TyDiQA-id, inline, answer-aware)

TyDiQA Gold Passage → Indonesian subset → `(input, target)` where
`input = "buat pertanyaan: {answer sentence with the answer span wrapped in <hl> … <hl>}"`
and `target = the question`. Highlighting the answer is what makes the model answer-aware.

In [ ]:
import re
from datasets import load_dataset, Dataset

PREFIX = "buat pertanyaan: "
SENT = re.compile(r"(?<=[.!?])\s+")

def answer_sentence(ctx, ans, start):
    if start is None or start < 0 or ctx[start:start + len(ans)] != ans:
        start = max(ctx.find(ans), 0)
    end = start + len(ans)
    s = 0
    for m in SENT.finditer(ctx):
        if m.end() <= start: s = m.end()
        else: break
    e = len(ctx)
    for m in SENT.finditer(ctx):
        if m.start() >= end: e = m.start(); break
    return ctx[s:e].strip()

def highlight_answer(sentence, ans):
    """Wrap the answer span with <hl> so the model learns ANSWER-AWARE QG:
    it is told which span to ask about, instead of guessing from the sentence.
    The format MUST match the HF Space inference side
    (backend/asahlagi-quizgen/qg_core.py -> answer_aware_prompt): "<hl> {ans} <hl>"."""
    i = sentence.find(ans)
    if i < 0:
        return sentence  # answer not found verbatim (rare) -> leave unhighlighted
    return f"{sentence[:i]}<hl> {ans} <hl>{sentence[i + len(ans):]}"

def build(split):
    rows = []
    for ex in split:
        if not str(ex["id"]).lower().startswith("indonesian"):
            continue
        ctx, q = ex["context"].strip(), ex["question"].strip()
        texts, starts = ex["answers"]["text"], ex["answers"]["answer_start"]
        if not (ctx and q and texts):
            continue
        ans = texts[0].strip()
        st = starts[0] if starts else ctx.find(ans)
        if not ans:
            continue
        sent = answer_sentence(ctx, ans, st)
        rows.append({"input": PREFIX + highlight_answer(sent, ans), "target": q})
    return rows

raw = load_dataset("tydiqa", "secondary_task")
train_rows, val_rows = build(raw["train"]), build(raw["validation"])
print("train:", len(train_rows), "| val:", len(val_rows))
print("sample input:", train_rows[0]["input"][:160])
print("sample target:", train_rows[0]["target"])

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

## 3. Load tokenizer and base model

In [ ]:
from transformers import T5TokenizerFast, T5ForConditionalGeneration

MODEL = "Wikidepia/IndoT5-base"
tok = T5TokenizerFast.from_pretrained(MODEL)
# Answer-aware QG: <hl> brackets the answer span in the input. Register it as a
# single special token so the highlight is one clean, consistent signal (not
# split into "<", "hl", ">") and so tokenisation matches the HF Space inference
# side, which loads this same pushed tokenizer.
tok.add_special_tokens({"additional_special_tokens": ["<hl>"]})

model = T5ForConditionalGeneration.from_pretrained(MODEL)
model.resize_token_embeddings(len(tok))  # add an embedding row for <hl>
print("params (M):", sum(p.numel() for p in model.parameters()) // 1_000_000, "| vocab:", len(tok))

## 4. Tokenize

In [ ]:
MAX_IN, MAX_OUT = 256, 64

def prep(batch):
    x = tok(batch["input"], max_length=MAX_IN, truncation=True)
    y = tok(text_target=batch["target"], max_length=MAX_OUT, truncation=True)
    x["labels"] = y["input_ids"]
    return x

train_tok = train_ds.map(prep, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(prep, batched=True, remove_columns=val_ds.column_names)

## 5. Train to 8 epochs (checkpoint every epoch)

`save_strategy="epoch"` + `save_total_limit=None` keeps a checkpoint per epoch in
`indot5-quizgen/checkpoint-*`, which section 7 uses to compare epochs 4-8.
`fp16=False` is the critical fix.

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)

MAX_EPOCHS = 8

args = Seq2SeqTrainingArguments(
    output_dir="indot5-quizgen",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=MAX_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=None,   # keep every epoch's checkpoint for the sweep
    predict_with_generate=True,
    fp16=False,   # MUST stay False (T5 + fp16 -> NaN)
    bf16=False,
    logging_steps=50,
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tok, model=model)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=collator, tokenizer=tok,
)
trainer.train()

## 6. Quick test (final model)

In [ ]:
def gen(sentence, answer):
    """Answer-aware generation: highlight `answer` so the model asks about it."""
    text = sentence.replace(answer, f"<hl> {answer} <hl>", 1)
    ids = tok(PREFIX + text, return_tensors="pt", truncation=True,
              max_length=MAX_IN).input_ids.to(model.device)
    return tok.decode(model.generate(ids, max_length=MAX_OUT, num_beams=4)[0],
                      skip_special_tokens=True)

# The generated question should target the highlighted answer.
print(gen("Fotosintesis adalah proses pembentukan glukosa oleh tumbuhan hijau dengan bantuan cahaya matahari.", "glukosa"))
print(gen("Proklamasi kemerdekaan Indonesia dibacakan oleh Soekarno pada tanggal 17 Agustus 1945.", "Soekarno"))
print(gen("Gunung Merapi terletak di perbatasan Jawa Tengah dan Yogyakarta.", "Merapi"))

## 7. Compare base vs epochs 4-8 (side-by-side, in-notebook)

7a plots eval loss for all 8 epochs. 7b loads the per-epoch checkpoints and builds one
table: Materi vs Base vs Ep4..Ep8. Pick the epoch where questions read best and the eval
loss has not started rising.

In [ ]:
# 7a. Eval-loss per epoch
import pandas as pd
from IPython.display import display

evals = [{"epoch": round(h["epoch"]), "eval_loss": round(h["eval_loss"], 4)}
         for h in trainer.state.log_history if "eval_loss" in h]
eval_df = pd.DataFrame(evals)
print("Eval loss per epoch (lowest = best fit; rising = overfitting):")
display(eval_df)
try:
    ax = eval_df.plot(x="epoch", y="eval_loss", marker="o",
                      title="Eval loss per epoch", figsize=(5, 3), legend=False)
    ax.set_ylabel("eval_loss")
except Exception as e:
    print("(plot skipped:", e, ")")

In [ ]:
# 7b. Side-by-side sweep: Base + epochs 4-8 (answer-aware input)
import os, json, gc, torch, pandas as pd
from IPython.display import display

EPOCHS_TO_COMPARE = [4, 5, 6, 7, 8]
# (passage, answer-to-highlight) — the question should target the answer.
tests = [
    ("Fotosintesis adalah proses pembentukan glukosa oleh tumbuhan hijau dengan bantuan cahaya matahari dan klorofil.", "glukosa"),
    ("Proklamasi kemerdekaan Indonesia dibacakan oleh Soekarno pada tanggal 17 Agustus 1945 di Jakarta.", "Soekarno"),
    ("Gunung Merapi adalah gunung berapi paling aktif di Indonesia, di perbatasan Jawa Tengah dan Yogyakarta.", "Merapi"),
    ("Jantung manusia memompa darah ke seluruh tubuh melalui pembuluh darah arteri dan vena.", "darah"),
    ("Python diciptakan oleh Guido van Rossum dan dirilis pertama kali pada tahun 1991.", "1991"),
    ("Hujan terjadi karena uap air di atmosfer mengembun menjadi titik air yang jatuh ke bumi.", "uap air"),
]

def gen_all(m, tk):
    res = []
    for passage, ans in tests:
        text = passage.replace(ans, f"<hl> {ans} <hl>", 1)
        ids = tk(PREFIX + text, return_tensors="pt", truncation=True, max_length=MAX_IN).input_ids.to(m.device)
        res.append(tk.decode(m.generate(ids, max_length=MAX_OUT, num_beams=4)[0], skip_special_tokens=True))
    return res

# map epoch -> checkpoint dir (robust: read each checkpoint's trainer_state.json)
ck_root = "indot5-quizgen"
epoch_to_ckpt = {}
for d in os.listdir(ck_root):
    if not d.startswith("checkpoint-"):
        continue
    st = json.load(open(os.path.join(ck_root, d, "trainer_state.json")))
    epoch_to_ckpt[round(st["epoch"])] = os.path.join(ck_root, d)

cols = {}
# base (no <hl> token — shows why the un-highlighted base is weaker)
base_tok = T5TokenizerFast.from_pretrained("Wikidepia/IndoT5-base")
bm = T5ForConditionalGeneration.from_pretrained("Wikidepia/IndoT5-base").to(model.device).eval()
cols["Base"] = gen_all(bm, base_tok)
del bm; gc.collect(); torch.cuda.empty_cache()

# per-epoch checkpoints (reuse `tok` — it carries the <hl> special token)
for ep in EPOCHS_TO_COMPARE:
    if ep not in epoch_to_ckpt:
        print(f"(epoch {ep} checkpoint not found, skipping)")
        continue
    m = T5ForConditionalGeneration.from_pretrained(epoch_to_ckpt[ep]).to(model.device).eval()
    cols[f"Ep{ep}"] = gen_all(m, tok)
    del m; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame({"Materi": [f"{p[:34]}… [{a}]" for p, a in tests], **cols})
styled = (df.style
          .set_properties(**{"text-align": "left", "white-space": "normal", "vertical-align": "top"})
          .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
display(styled)

## 8. Push the chosen model to Hugging Face Hub

Decide the best epoch from section 7. If it is the **final** epoch (8), the in-memory `model`
is already it. Otherwise load that epoch's checkpoint first:

```python
# example: use epoch 6 instead of the final model
model = T5ForConditionalGeneration.from_pretrained(epoch_to_ckpt[6]).to(model.device)
```

Then paste a **WRITE** token and push.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
REPO = "raviarnan/indot5-quizgen-asahlagi"
model.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("pushed to", REPO)

## 9. Deploy the fine-tuned model to the quiz-gen Space

The Space (`backend/asahlagi-quizgen/`) **already** points at this repo and **already** does
answer-aware inference + cloze fallback (`app.py` + `qg_core.py`, model id
`raviarnan/indot5-quizgen-asahlagi`). So once `push_to_hub` above finishes:

1. Open the Space on Hugging Face → **Settings → Factory rebuild** (picks up the new weights),
   or just wait for the next rebuild.
2. No code change needed — the prefix and `<hl> {answer} <hl>` format already match.
3. Verify: `GET /` shows `status: ready`, then `POST /generate` — questions should now read
   naturally and genuinely target the highlighted answer.

> The push overwrites the previous (plain-QG) weights at the same repo id, so the Space
> upgrades in place. The `<hl>` special token rides along in the pushed tokenizer.